# Preprocessing — Colab runner (A/B the label schemes)

Runs `preprocess.py` on your **generated** data. Each of the three label schemes — `cause`, `obs_or`, `obs_and` — runs in its **own cell** and writes to its **own** `processed_<scheme>/` folder, so you can run and test each individually without overwriting the others.

Outputs per scheme: `feature_table.parquet`, `windows_*.npz`, `class_weights.json`, `split_manifest.csv`. **Edit the PAT prompt and `DATA` if needed.**

In [ ]:
import os
import getpass

REPO_OWNER = "savvy-sam"
REPO_NAME = "sewer-blockage"

PAT = os.environ.get("GITHUB_PAT") or getpass.getpass("Enter your GitHub PAT: ")
REPO_URL = f"https://{PAT}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

name = REPO_NAME

if os.path.isdir(name):
    !cd {name} && git remote set-url origin {REPO_URL} && git pull --ff-only
else:
    !git clone {REPO_URL}

%cd {name}

In [ ]:
!pip -q install -r requirements.txt scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Shared config

`DATA` = the folder holding `runs/*.parquet` + `manifest.csv` (your generated data). Same window/stride for all schemes so the only thing that differs is the label.

In [ ]:
DATA = '/content/drive/MyDrive/data_generation/data'   # <-- folder with runs/ + manifest.csv
WINDOW, STRIDE = 60, 10

## Scheme A — `cause`
Blockage = the clog is switched on, whether or not a sensor could see it (the strict/original baseline).

In [ ]:
!python preprocess.py --data '{DATA}' --out processed_cause --label-col cause --window {WINDOW} --stride {STRIDE}

## Scheme B — `obs_or`
Blockage = visible on **either** the upstream level **or** the downstream flow sensor.

In [ ]:
!python preprocess.py --data '{DATA}' --out processed_obs_or --label-col obs_or --window {WINDOW} --stride {STRIDE}

## Scheme C — `obs_and`  (default)
Blockage = visible on **both** sensors (agreement — highest confidence).

In [ ]:
!python preprocess.py --data '{DATA}' --out processed_obs_and --label-col obs_and --window {WINDOW} --stride {STRIDE}

## Inspect a scheme
Set `SCHEME` to whichever folder you want to look at.

In [ ]:
import pandas as pd, json, glob, os

SCHEME = 'obs_and'          # <-- change to 'cause' | 'obs_or' | 'obs_and' to inspect each
P = f'processed_{SCHEME}'

sm = pd.read_csv(f'{P}/split_manifest.csv')
print('scheme:', SCHEME)
print('\nrun-level split:'); print(sm['split'].value_counts().to_string())
ft = pd.read_parquet(f'{P}/feature_table.parquet')
print('\nfeature_table:', ft.shape)
print('\nclass balance per split (timesteps):')
print(pd.crosstab(ft['split'], ft['label']).to_string())
print('\nwindow files:', [os.path.basename(f) for f in sorted(glob.glob(f'{P}/windows_*.npz'))])
print('class weights (train):', json.load(open(f'{P}/class_weights.json')))

## Save all schemes to Drive
Copies each `processed_<scheme>/` you have run to its own Drive folder (training reads from there).

In [ ]:
import os
for sch in ['cause', 'obs_or', 'obs_and']:
    src  = f'processed_{sch}'
    dest = f'/content/drive/MyDrive/data_generation/data/processed_{sch}'
    if os.path.isdir(src):
        os.makedirs(dest, exist_ok=True)
        !cp -r {src}/* '{dest}/'
        print('saved ->', dest)
    else:
        print('skip (not run yet):', src)